In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Any, Callable, Iterator

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import Dataset

In [2]:

def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_metrics_json(metrics: dict[str, Any], output_path: str | Path) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)


def plot_training_curves(
    loss_hist: list[float],
    eval_steps: list[int],
    val_psnr_hist: list[float],
    output_dir: str | Path,
) -> None:
    output_dir = ensure_dir(output_dir)

    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    ax.plot(loss_hist)
    ax.set_title("Training Loss")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_dir / "loss_curve.png", dpi=150)
    plt.close(fig)

    if eval_steps and val_psnr_hist:
        fig, ax = plt.subplots(1, 1, figsize=(7, 4))
        ax.plot(eval_steps, val_psnr_hist)
        ax.set_title("Validation PSNR")
        ax.set_xlabel("Step")
        ax.set_ylabel("PSNR (dB)")
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(output_dir / "psnr_curve.png", dpi=150)
        plt.close(fig)


def save_rgb_png(image: np.ndarray, output_path: str | Path) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    img = np.clip(image, 0.0, 1.0)
    img_uint8 = (img * 255.0).round().astype(np.uint8)
    Image.fromarray(img_uint8, mode="RGB").save(output_path)


def save_depth_png(depth: np.ndarray, output_path: str | Path) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    d = depth.astype(np.float32)
    d_min = float(np.min(d))
    d_max = float(np.max(d))
    if d_max > d_min:
        d = (d - d_min) / (d_max - d_min)
    else:
        d = np.zeros_like(d)

    d_rgba = plt.cm.inferno(d)
    d_rgb_uint8 = (d_rgba[..., :3] * 255.0).round().astype(np.uint8)
    Image.fromarray(d_rgb_uint8, mode="RGB").save(output_path)


def load_data(data_path: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, torch.Tensor]:
    data = np.load(data_path)
    images_train = data["images_train"] / 255.0
    c2ws_train = data["c2ws_train"]
    images_val = data["images_val"] / 255.0
    c2ws_val = data["c2ws_val"]
    c2ws_test = data["c2ws_test"]
    focal = data["focal"]

    h, w = images_train.shape[1], images_train.shape[2]
    o_x = w / 2
    o_y = h / 2
    k = torch.as_tensor([[focal.item(), 0, o_x], [0, focal.item(), o_y], [0, 0, 1]])
    return images_train, c2ws_train, images_val, c2ws_val, c2ws_test, k


def pixels_to_rays(
    k: torch.Tensor,
    c2w: torch.Tensor,
    uvs: torch.Tensor,
    device: str = "cuda",
) -> tuple[torch.Tensor, torch.Tensor]:
    k = k.to(device)
    c2w = c2w.to(torch.float64).to(device)
    uvs = uvs.to(device)

    num_pixels = uvs.shape[0]
    r = c2w[:3, :3]
    r_os = c2w[:3, 3].unsqueeze(0).expand(num_pixels, -1)

    homog_uvs = torch.hstack((uvs, torch.ones(num_pixels, 1, device=device)))
    k_inv = torch.linalg.inv(k).to(device)
    dirs = ((r @ k_inv.to(torch.float64)) @ homog_uvs.to(torch.float64).T).T
    r_ds = dirs / torch.linalg.norm(dirs, dim=1, keepdim=True)
    return r_os, r_ds


def image_to_rays(
    image: torch.Tensor,
    c2w: torch.Tensor,
    k: torch.Tensor,
    device: str = "cuda",
) -> torch.Tensor:
    image = image.to(device)
    h, w = image.shape[:2]
    ys, xs = torch.meshgrid(
        torch.arange(h, device=device),
        torch.arange(w, device=device),
        indexing="ij",
    )
    uvs = torch.stack((xs.reshape(-1), ys.reshape(-1)), dim=-1).to(torch.float32)
    r_os, r_ds = pixels_to_rays(k=k, c2w=c2w, uvs=uvs, device=device)
    return torch.cat((r_os, r_ds), dim=1).reshape(h, w, 6)


def images_to_rays(
    images: torch.Tensor,
    c2ws: torch.Tensor,
    k: torch.Tensor,
    device: str = "cuda",
) -> torch.Tensor:
    rays_per_image = [image_to_rays(images[i], c2ws[i], k, device=device) for i in range(images.shape[0])]
    return torch.stack(rays_per_image, dim=0)


class RaysData(Dataset):
    def __init__(
        self,
        images: torch.Tensor,
        k: torch.Tensor,
        c2ws: torch.Tensor,
        device: str = "cuda",
    ) -> None:
        self.images = torch.as_tensor(images, dtype=torch.float32, device=device)
        self.k = torch.as_tensor(k, dtype=torch.float32, device=device)
        self.c2ws = torch.as_tensor(c2ws, dtype=torch.float32, device=device)
        self.h, self.w = self.images.shape[1:3]
        self.num_images = self.images.shape[0]

        ys, xs = torch.meshgrid(
            torch.arange(self.h, device=device),
            torch.arange(self.w, device=device),
            indexing="ij",
        )
        single_image_uvs = torch.stack((xs.reshape(-1), ys.reshape(-1)), dim=-1)
        self.uvs = single_image_uvs.repeat(self.num_images, 1)

        rays = images_to_rays(self.images, self.c2ws, self.k, device=device)
        self.rays_o = rays[..., :3].reshape(-1, 3)
        self.rays_d = rays[..., 3:].reshape(-1, 3)
        self.gt_rgbs = self.images.reshape(-1, 3)

    def __len__(self) -> int:
        return self.num_images * self.h * self.w

    def sample_rays(self, num_rays: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        sample_indices = torch.randint(0, len(self), (num_rays,), device=self.rays_o.device)
        return (
            self.rays_o[sample_indices],
            self.rays_d[sample_indices],
            self.gt_rgbs[sample_indices],
        )


def positional_encoding(x: torch.Tensor, num_freqs: int) -> torch.Tensor:
    if num_freqs <= 0:
        return x
    enc = [x]
    freq_bands = 2.0 ** torch.arange(num_freqs, device=x.device, dtype=x.dtype)
    for freq in freq_bands:
        enc.append(torch.sin(freq * x))
        enc.append(torch.cos(freq * x))
    return torch.cat(enc, dim=-1)


class NeRFMLP(nn.Module):
    def __init__(
        self,
        pos_freqs: int = 10,
        dir_freqs: int = 4,
        hidden_dim: int = 256,
        n_layers: int = 8,
        skip_layer: int = 4,
    ) -> None:
        super().__init__()
        if n_layers < 2:
            raise ValueError("n_layers must be >= 2")

        self.pos_freqs = pos_freqs
        self.dir_freqs = dir_freqs
        self.skip_layer = skip_layer

        pos_in_dim = 3 * (2 * pos_freqs + 1)
        dir_in_dim = 3 * (2 * dir_freqs + 1)

        self.pts_layers = nn.ModuleList()
        self.pts_layers.append(nn.Linear(pos_in_dim, hidden_dim))
        for i in range(1, n_layers):
            in_dim = hidden_dim + pos_in_dim if i == skip_layer else hidden_dim
            self.pts_layers.append(nn.Linear(in_dim, hidden_dim))

        self.sigma_head = nn.Linear(hidden_dim, 1)
        self.feature_head = nn.Linear(hidden_dim, hidden_dim)
        self.color_layers = nn.Sequential(
            nn.Linear(hidden_dim + dir_in_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, 3),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, xyz: torch.Tensor, ray_dirs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        param_dtype = self.pts_layers[0].weight.dtype
        param_device = self.pts_layers[0].weight.device
        xyz = xyz.to(device=param_device, dtype=param_dtype)
        ray_dirs = ray_dirs.to(device=param_device, dtype=param_dtype)

        n_rays, n_samples, _ = xyz.shape
        xyz_flat = xyz.reshape(-1, 3)
        dirs_expanded = ray_dirs[:, None, :].expand(-1, n_samples, -1).reshape(-1, 3)

        xyz_enc = positional_encoding(xyz_flat, self.pos_freqs)
        dir_enc = positional_encoding(dirs_expanded, self.dir_freqs)

        h = xyz_enc
        for i, layer in enumerate(self.pts_layers):
            if i == self.skip_layer:
                h = torch.cat([h, xyz_enc], dim=-1)
            h = self.relu(layer(h))

        sigma = self.sigma_head(h).reshape(n_rays, n_samples, 1)
        features = self.feature_head(h)
        rgb = self.color_layers(torch.cat([features, dir_enc], dim=-1)).reshape(n_rays, n_samples, 3)
        return sigma, rgb


def sample_along_rays(
    ray_origins: torch.Tensor,
    ray_directions: torch.Tensor,
    near: float,
    far: float,
    n_samples: int,
    perturb: bool = False,
) -> tuple[torch.Tensor, torch.Tensor]:
    if n_samples < 2:
        raise ValueError("n_samples must be >= 2")

    t_lin = torch.linspace(near, far, n_samples, device=ray_origins.device, dtype=ray_origins.dtype)
    t_vals = t_lin.unsqueeze(0).expand(ray_origins.shape[0], -1)

    if perturb:
        mids = 0.5 * (t_vals[:, :-1] + t_vals[:, 1:])
        lower = torch.cat([t_vals[:, :1], mids], dim=-1)
        upper = torch.cat([mids, t_vals[:, -1:]], dim=-1)
        t_vals = lower + (upper - lower) * torch.rand_like(t_vals)

    xyz_samples = ray_origins[:, None, :] + ray_directions[:, None, :] * t_vals[..., None]
    return xyz_samples, t_vals


def sample_pdf(
    bins: torch.Tensor,
    weights: torch.Tensor,
    n_importance: int,
    deterministic: bool = False,
) -> torch.Tensor:
    eps = 1e-5
    weights = weights + eps
    pdf = weights / torch.sum(weights, dim=-1, keepdim=True)
    cdf = torch.cumsum(pdf, dim=-1)
    cdf = torch.cat([torch.zeros_like(cdf[:, :1]), cdf], dim=-1)

    if deterministic:
        u = torch.linspace(0.0, 1.0, n_importance, device=bins.device, dtype=bins.dtype).expand(cdf.shape[0], -1)
    else:
        u = torch.rand(cdf.shape[0], n_importance, device=bins.device, dtype=bins.dtype)

    inds = torch.searchsorted(cdf.contiguous(), u.contiguous(), right=True)
    below = torch.clamp_min(inds - 1, 0)
    above = torch.clamp_max(inds, cdf.shape[-1] - 1)

    cdf_g0 = torch.gather(cdf, 1, below)
    cdf_g1 = torch.gather(cdf, 1, above)

    bins_pad = torch.cat([bins[:, :1], bins], dim=-1)
    bins_g0 = torch.gather(bins_pad, 1, below)
    bins_g1 = torch.gather(bins_pad, 1, above)

    denom = torch.where((cdf_g1 - cdf_g0) < eps, torch.ones_like(cdf_g1), cdf_g1 - cdf_g0)
    return bins_g0 + ((u - cdf_g0) / denom) * (bins_g1 - bins_g0)


def volume_render(
    sigmas: torch.Tensor,
    rgbs: torch.Tensor,
    t_vals: torch.Tensor,
    ray_directions: torch.Tensor,
    white_bkgd: bool = False,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    sigma = torch.relu(sigmas.squeeze(-1))
    dists = t_vals[:, 1:] - t_vals[:, :-1]
    dists = torch.cat([dists, torch.full_like(dists[:, :1], 1e10)], dim=-1)
    dists = dists * torch.linalg.norm(ray_directions, dim=-1, keepdim=True)

    alpha = 1.0 - torch.exp(-sigma * dists)
    trans = torch.cumprod(
        torch.cat([torch.ones_like(alpha[:, :1]), 1.0 - alpha + 1e-10], dim=-1),
        dim=-1,
    )[:, :-1]
    weights = alpha * trans

    rgb_map = torch.sum(weights[..., None] * rgbs, dim=1)
    depth_map = torch.sum(weights * t_vals, dim=-1)
    acc_map = torch.sum(weights, dim=-1)

    if white_bkgd:
        rgb_map = rgb_map + (1.0 - acc_map)[..., None]

    return rgb_map, depth_map, acc_map, weights


def render_rays_hierarchical(
    ray_origins: torch.Tensor,
    ray_directions: torch.Tensor,
    model_coarse: Callable[[torch.Tensor, torch.Tensor], tuple[torch.Tensor, torch.Tensor]],
    model_fine: Callable[[torch.Tensor, torch.Tensor], tuple[torch.Tensor, torch.Tensor]] | None,
    n_coarse: int,
    n_fine: int,
    near: float,
    far: float,
    perturb: bool,
    white_bkgd: bool,
) -> dict[str, torch.Tensor]:
    param = next(model_coarse.parameters())
    ray_origins = ray_origins.to(device=param.device, dtype=param.dtype)
    ray_directions = ray_directions.to(device=param.device, dtype=param.dtype)

    xyz_c, t_c = sample_along_rays(ray_origins, ray_directions, near, far, n_coarse, perturb=perturb)
    sigma_c, rgb_c = model_coarse(xyz_c, ray_directions)
    rgb_map_c, depth_map_c, acc_map_c, weights_c = volume_render(
        sigma_c, rgb_c, t_c, ray_directions, white_bkgd=white_bkgd
    )

    out: dict[str, torch.Tensor] = {
        "rgb_coarse": rgb_map_c,
        "depth_coarse": depth_map_c,
        "acc_coarse": acc_map_c,
        "weights_coarse": weights_c,
        "t_coarse": t_c,
    }

    if model_fine is None or n_fine <= 0:
        out["rgb_fine"] = rgb_map_c
        out["depth_fine"] = depth_map_c
        out["acc_fine"] = acc_map_c
        return out

    t_mids = 0.5 * (t_c[:, :-1] + t_c[:, 1:])
    pdf_weights = weights_c[:, 1:-1].detach() + 1e-5
    t_fine = sample_pdf(t_mids, pdf_weights, n_importance=n_fine, deterministic=not perturb)
    t_all = torch.sort(torch.cat([t_c, t_fine], dim=-1), dim=-1).values
    xyz_f = ray_origins[:, None, :] + ray_directions[:, None, :] * t_all[..., None]

    sigma_f, rgb_f = model_fine(xyz_f, ray_directions)
    rgb_map_f, depth_map_f, acc_map_f, weights_f = volume_render(
        sigma_f, rgb_f, t_all, ray_directions, white_bkgd=white_bkgd
    )

    out.update(
        {
            "rgb_fine": rgb_map_f,
            "depth_fine": depth_map_f,
            "acc_fine": acc_map_f,
            "weights_fine": weights_f,
            "t_fine": t_all,
        }
    )
    return out


@torch.no_grad()
def render_image_by_chunks(
    ray_origins: torch.Tensor,
    ray_directions: torch.Tensor,
    render_fn: Callable[[torch.Tensor, torch.Tensor], dict[str, torch.Tensor]],
    chunk_size: int,
) -> dict[str, torch.Tensor]:
    outputs: dict[str, list[torch.Tensor]] = {}
    for start in range(0, ray_origins.shape[0], chunk_size):
        end = min(start + chunk_size, ray_origins.shape[0])
        chunk_out = render_fn(ray_origins[start:end], ray_directions[start:end])
        for key, val in chunk_out.items():
            outputs.setdefault(key, []).append(val)
    return {key: torch.cat(vals, dim=0) for key, vals in outputs.items()}


def build_part2_data(data_path: str, device: str) -> dict[str, Any]:
    images_train, c2ws_train, images_val, c2ws_val, c2ws_test, k = load_data(data_path)

    train_images_t = torch.as_tensor(images_train, dtype=torch.float32, device=device)
    train_c2ws_t = torch.as_tensor(c2ws_train, dtype=torch.float32, device=device)
    val_images_t = torch.as_tensor(images_val, dtype=torch.float32, device=device)
    val_c2ws_t = torch.as_tensor(c2ws_val, dtype=torch.float32, device=device)
    test_c2ws_t = torch.as_tensor(c2ws_test, dtype=torch.float32, device=device)
    k_t = torch.as_tensor(k, dtype=torch.float32, device=device)

    train_dataset = RaysData(train_images_t, k_t, train_c2ws_t, device=device)
    return {
        "k": k_t,
        "train_dataset": train_dataset,
        "val_images": val_images_t,
        "val_c2ws": val_c2ws_t,
        "test_c2ws": test_c2ws_t,
    }


def calibrate_near_far_from_cameras(c2ws: torch.Tensor) -> tuple[float, float]:
    centers = c2ws[:, :3, 3]
    dists = torch.linalg.norm(centers, dim=-1)
    q10 = torch.quantile(dists, 0.10).item()
    q90 = torch.quantile(dists, 0.90).item()
    near = max(0.1, 0.5 * q10)
    far = max(near + 1.0, 1.5 * q90)
    return near, far


def mse_to_psnr(mse_val: torch.Tensor | float) -> float:
    mse = torch.tensor(mse_val, dtype=torch.float32) if not isinstance(mse_val, torch.Tensor) else mse_val.float()
    mse = torch.clamp(mse, min=1e-12)
    return float(-10.0 * torch.log10(mse).item())


@torch.no_grad()
def render_full_validation_image(
    image: torch.Tensor,
    c2w: torch.Tensor,
    k: torch.Tensor,
    model_coarse: NeRFMLP,
    model_fine: NeRFMLP,
    near: float,
    far: float,
    n_coarse: int,
    n_fine: int,
    chunk_size: int,
    device: str,
) -> tuple[torch.Tensor, float]:
    h, w = image.shape[:2]
    rays = image_to_rays(image, c2w, k, device=device)
    rays_o = rays[..., :3].reshape(-1, 3).float()
    rays_d = rays[..., 3:].reshape(-1, 3).float()

    def _render_fn(chunk_o: torch.Tensor, chunk_d: torch.Tensor) -> dict[str, torch.Tensor]:
        return render_rays_hierarchical(
            chunk_o,
            chunk_d,
            model_coarse=model_coarse,
            model_fine=model_fine,
            n_coarse=n_coarse,
            n_fine=n_fine,
            near=near,
            far=far,
            perturb=False,
            white_bkgd=False,
        )

    out = render_image_by_chunks(rays_o, rays_d, _render_fn, chunk_size=chunk_size)
    rgb_pred = out["rgb_fine"].reshape(h, w, 3)
    return rgb_pred, mse_to_psnr(F.mse_loss(rgb_pred, image))


def train_nerf_part2(
    data_path: str,
    output_dir: str,
    device: str = "cuda",
    seed: int = 42,
    hidden_dim: int = 256,
    n_layers: int = 8,
    pos_freqs: int = 10,
    dir_freqs: int = 4,
    n_coarse: int = 32,
    n_fine: int = 32,
    n_steps: int = 5000,
    batch_rays: int = 2048,
    lr: float = 5e-4,
    eval_every: int = 250,
    chunk_size: int = 4096,
    near_override: float | None = None,
    far_override: float | None = None,
    log_every: int | None = None,
    save_progress_renders: bool = True,
    verbose: bool = True,
) -> dict[str, Any]:
    set_seed(seed)
    out_path = ensure_dir(output_dir)

    data = build_part2_data(data_path, device=device)
    train_dataset: RaysData = data["train_dataset"]
    val_images: torch.Tensor = data["val_images"]
    val_c2ws: torch.Tensor = data["val_c2ws"]
    k: torch.Tensor = data["k"]

    near_auto, far_auto = calibrate_near_far_from_cameras(val_c2ws)
    near = near_override if near_override is not None else near_auto
    far = far_override if far_override is not None else far_auto
    log_every = max(1, n_steps // 20) if log_every is None else log_every

    model_coarse = NeRFMLP(pos_freqs=pos_freqs, dir_freqs=dir_freqs, hidden_dim=hidden_dim, n_layers=n_layers).to(device)
    model_fine = NeRFMLP(pos_freqs=pos_freqs, dir_freqs=dir_freqs, hidden_dim=hidden_dim, n_layers=n_layers).to(device)
    optimizer = torch.optim.Adam(list(model_coarse.parameters()) + list(model_fine.parameters()), lr=lr)

    loss_hist: list[float] = []
    eval_steps: list[int] = []
    val_psnr_hist: list[float] = []
    best_psnr = -1.0
    best_step = -1
    run_start = time.perf_counter()

    if verbose:
        print(
            f"[train_nerf_part2] start device={device} steps={n_steps} "
            f"batch_rays={batch_rays} coarse={n_coarse} fine={n_fine} "
            f"near={near:.3f} far={far:.3f} eval_every={eval_every}"
        )

    for step in range(1, n_steps + 1):
        iter_start = time.perf_counter()
        rays_o, rays_d, gt_rgb = train_dataset.sample_rays(batch_rays)
        out = render_rays_hierarchical(
            rays_o.float(),
            rays_d.float(),
            model_coarse=model_coarse,
            model_fine=model_fine,
            n_coarse=n_coarse,
            n_fine=n_fine,
            near=near,
            far=far,
            perturb=True,
            white_bkgd=False,
        )

        loss_coarse = F.mse_loss(out["rgb_coarse"], gt_rgb.float())
        loss_fine = F.mse_loss(out["rgb_fine"], gt_rgb.float())
        loss = 0.1 * loss_coarse + loss_fine

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        loss_hist.append(float(loss.item()))

        if verbose and (step == 1 or step % log_every == 0 or step == n_steps):
            iter_seconds = time.perf_counter() - iter_start
            rays_per_sec = batch_rays / max(iter_seconds, 1e-9)
            print(
                f"[train_nerf_part2] step {step}/{n_steps} "
                f"loss={loss.item():.6f} coarse={loss_coarse.item():.6f} "
                f"fine={loss_fine.item():.6f} iter={iter_seconds:.3f}s "
                f"rays_per_sec={rays_per_sec:,.0f}"
            )

        if step % eval_every == 0 or step == n_steps:
            eval_start = time.perf_counter()
            val_rgb_pred, val_psnr = render_full_validation_image(
                image=val_images[0],
                c2w=val_c2ws[0],
                k=k,
                model_coarse=model_coarse,
                model_fine=model_fine,
                near=near,
                far=far,
                n_coarse=n_coarse,
                n_fine=n_fine,
                chunk_size=chunk_size,
                device=device,
            )
            eval_seconds = time.perf_counter() - eval_start

            if save_progress_renders:
                progress_dir = ensure_dir(out_path / "progress_renders")
                save_rgb_png(val_rgb_pred.cpu().numpy(), progress_dir / f"step_{step:04d}.png")

            eval_steps.append(step)
            val_psnr_hist.append(val_psnr)

            if val_psnr > best_psnr:
                best_psnr = val_psnr
                best_step = step
                torch.save(
                    {
                        "model_coarse": model_coarse.state_dict(),
                        "model_fine": model_fine.state_dict(),
                        "step": step,
                        "best_psnr": best_psnr,
                    },
                    out_path / "checkpoint_best.pt",
                )
                if verbose:
                    print(
                        f"[train_nerf_part2] eval step {step}/{n_steps} "
                        f"val_psnr={val_psnr:.3f}dB best={best_psnr:.3f}dB "
                        f"(new best, saved checkpoint) eval={eval_seconds:.2f}s"
                    )
            elif verbose:
                print(
                    f"[train_nerf_part2] eval step {step}/{n_steps} "
                    f"val_psnr={val_psnr:.3f}dB best={best_psnr:.3f}dB "
                    f"eval={eval_seconds:.2f}s"
                )

    total_seconds = time.perf_counter() - run_start
    metrics = {
        "best_psnr": best_psnr,
        "best_step": best_step,
        "final_loss": loss_hist[-1] if loss_hist else None,
        "near": near,
        "far": far,
        "n_steps": n_steps,
        "batch_rays": batch_rays,
        "n_coarse": n_coarse,
        "n_fine": n_fine,
        "val_psnr_hist": val_psnr_hist,
        "eval_steps": eval_steps,
        "total_seconds": total_seconds,
        "avg_seconds_per_step": total_seconds / max(n_steps, 1),
        "log_every": log_every,
    }
    save_metrics_json(metrics, out_path / "report_metrics.json")

    if verbose:
        print(
            f"[train_nerf_part2] done total={total_seconds:.2f}s "
            f"avg_step={total_seconds / max(n_steps, 1):.4f}s "
            f"best_psnr={best_psnr:.3f}dB step={best_step}"
        )

    return {
        "metrics": metrics,
        "loss_hist": loss_hist,
        "eval_steps": eval_steps,
        "val_psnr_hist": val_psnr_hist,
        "models": {"coarse": model_coarse, "fine": model_fine},
        "data": data,
    }


@torch.no_grad()
def _iter_test_trajectory_renders(
    model_coarse: NeRFMLP,
    model_fine: NeRFMLP,
    k: torch.Tensor,
    test_c2ws: torch.Tensor,
    image_hw: tuple[int, int],
    near: float,
    far: float,
    n_coarse: int,
    n_fine: int,
    chunk_size: int,
    device: str,
) -> Iterator[tuple[int, np.ndarray, np.ndarray]]:
    h, w = image_hw
    dummy_image = torch.zeros((h, w, 3), device=device)

    for idx in range(test_c2ws.shape[0]):
        print(f"Rendering test view {idx + 1}/{test_c2ws.shape[0]}...")
        rays = image_to_rays(dummy_image, test_c2ws[idx], k, device=device)
        rays_o = rays[..., :3].reshape(-1, 3).float()
        rays_d = rays[..., 3:].reshape(-1, 3).float()

        def _render_fn(chunk_o: torch.Tensor, chunk_d: torch.Tensor) -> dict[str, torch.Tensor]:
            return render_rays_hierarchical(
                chunk_o,
                chunk_d,
                model_coarse=model_coarse,
                model_fine=model_fine,
                n_coarse=n_coarse,
                n_fine=n_fine,
                near=near,
                far=far,
                perturb=False,
                white_bkgd=False,
            )

        out = render_image_by_chunks(rays_o, rays_d, _render_fn, chunk_size=chunk_size)
        rgb = out["rgb_fine"].reshape(h, w, 3).detach().cpu().numpy()
        depth = out["depth_fine"].reshape(h, w).detach().cpu().numpy()
        yield idx, rgb, depth


def _save_gif_from_pngs(png_paths: list[Path], output_path: str | Path, duration: float) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames = [imageio.imread(path) for path in png_paths]
    imageio.mimsave(output_path, frames, duration=duration)


@torch.no_grad()
def render_test_trajectory_rgb(
    model_coarse: NeRFMLP,
    model_fine: NeRFMLP,
    k: torch.Tensor,
    test_c2ws: torch.Tensor,
    image_hw: tuple[int, int],
    output_dir: str,
    near: float,
    far: float,
    n_coarse: int,
    n_fine: int,
    chunk_size: int,
    device: str,
    gif_duration: float = 0.1,
) -> None:
    out_path = Path(output_dir)
    rgb_npy_dir = ensure_dir(out_path / "test_rgb_npy")
    rgb_png_dir = ensure_dir(out_path / "test_rgb_png")

    png_paths: list[Path] = []
    for idx, rgb, _depth in _iter_test_trajectory_renders(
        model_coarse, model_fine, k, test_c2ws, image_hw, near, far, n_coarse, n_fine, chunk_size, device
    ):
        np.save(rgb_npy_dir / f"view_{idx:02d}.npy", rgb)
        png_path = rgb_png_dir / f"view_{idx:02d}.png"
        save_rgb_png(rgb, png_path)
        png_paths.append(png_path)

    _save_gif_from_pngs(png_paths, out_path / "test_rgb.gif", duration=gif_duration)


@torch.no_grad()
def render_test_trajectory_depth(
    model_coarse: NeRFMLP,
    model_fine: NeRFMLP,
    k: torch.Tensor,
    test_c2ws: torch.Tensor,
    image_hw: tuple[int, int],
    output_dir: str,
    near: float,
    far: float,
    n_coarse: int,
    n_fine: int,
    chunk_size: int,
    device: str,
    gif_duration: float = 0.1,
) -> None:
    out_path = Path(output_dir)
    depth_npy_dir = ensure_dir(out_path / "test_depth_npy")
    depth_png_dir = ensure_dir(out_path / "test_depth_png")

    png_paths: list[Path] = []
    for idx, _rgb, depth in _iter_test_trajectory_renders(
        model_coarse, model_fine, k, test_c2ws, image_hw, near, far, n_coarse, n_fine, chunk_size, device
    ):
        np.save(depth_npy_dir / f"view_{idx:02d}.npy", depth)
        png_path = depth_png_dir / f"view_{idx:02d}.png"
        save_depth_png(depth, png_path)
        png_paths.append(png_path)

    _save_gif_from_pngs(png_paths, out_path / "test_depth.gif", duration=gif_duration)


def run_part2_pipeline(
    cfg: dict[str, Any],
    render_rgb: bool = True,
    render_depth: bool = True,
) -> dict[str, Any]:
    device = cfg.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    results = train_nerf_part2(
        data_path=cfg["data_path"],
        output_dir=cfg["output_dir"],
        device=device,
        seed=cfg.get("seed", 42),
        hidden_dim=cfg.get("hidden_dim", 256),
        n_layers=cfg.get("n_layers", 8),
        pos_freqs=cfg.get("pos_freqs", 10),
        dir_freqs=cfg.get("dir_freqs", 4),
        n_coarse=cfg.get("n_coarse", 32),
        n_fine=cfg.get("n_fine", 32),
        n_steps=cfg.get("n_steps", 5000),
        batch_rays=cfg.get("batch_rays", 2048),
        lr=cfg.get("lr", 5e-4),
        eval_every=cfg.get("eval_every", 250),
        chunk_size=cfg.get("chunk_size", 4096),
        near_override=cfg.get("near_override"),
        far_override=cfg.get("far_override"),
        log_every=cfg.get("log_every"),
        save_progress_renders=cfg.get("save_progress_renders", True),
        verbose=cfg.get("verbose", True),
    )

    plot_training_curves(
        results["loss_hist"],
        results["eval_steps"],
        results["val_psnr_hist"],
        cfg["output_dir"],
    )

    models = results["models"]
    data = results["data"]
    image_hw = (data["val_images"].shape[1], data["val_images"].shape[2])

    if render_rgb:
        render_test_trajectory_rgb(
            model_coarse=models["coarse"],
            model_fine=models["fine"],
            k=data["k"],
            test_c2ws=data["test_c2ws"],
            image_hw=image_hw,
            output_dir=cfg["output_dir"],
            near=results["metrics"]["near"],
            far=results["metrics"]["far"],
            n_coarse=cfg.get("n_coarse", 32),
            n_fine=cfg.get("n_fine", 32),
            chunk_size=cfg.get("chunk_size", 4096),
            device=device,
            gif_duration=cfg.get("gif_duration", 0.1),
        )

    if render_depth:
        render_test_trajectory_depth(
            model_coarse=models["coarse"],
            model_fine=models["fine"],
            k=data["k"],
            test_c2ws=data["test_c2ws"],
            image_hw=image_hw,
            output_dir=cfg["output_dir"],
            near=results["metrics"]["near"],
            far=results["metrics"]["far"],
            n_coarse=cfg.get("n_coarse", 32),
            n_fine=cfg.get("n_fine", 32),
            chunk_size=cfg.get("chunk_size", 4096),
            device=device,
            gif_duration=cfg.get("gif_duration", 0.1),
        )

    return results


In [ ]:
import torch
import torch._utils  # Workaround for an internal PyTorch lazy-loading bug

default_cfg = {
        "seed": 42,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "data_path": "lego_200x200.npz",
        "output_dir": "images/output/part2_3d_reconstruction",
        "hidden_dim": 256,
        "n_layers": 8,
        "pos_freqs": 10,
        "dir_freqs": 4,
        "n_coarse": 32,
        "n_fine": 32,
        "batch_rays": 2048,
        "n_steps": 5000,
        "eval_every": 250,
        "chunk_size": 4096,
        "lr": 5e-4,
        "near_override": None,
        "far_override": None,
        "gif_duration": 0.1,
        "save_progress_renders": True,
        "verbose": True,
    }

run_part2_pipeline(default_cfg, render_rgb=True, render_depth=True)


[train_nerf_part2] start device=cuda steps=5000 batch_rays=2048 coarse=32 fine=32 near=2.016 far=6.047 eval_every=250
[train_nerf_part2] step 1/5000 loss=0.198122 coarse=0.091187 fine=0.189004 iter=0.893s rays_per_sec=2,295
[train_nerf_part2] step 250/5000 loss=0.012807 coarse=0.012965 fine=0.011510 iter=0.242s rays_per_sec=8,479


/tmp/ipykernel_5497/2785096391.py:56: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(img_uint8, mode="RGB").save(output_path)


[train_nerf_part2] eval step 250/5000 val_psnr=18.942dB best=18.942dB (new best, saved checkpoint) eval=1.90s
[train_nerf_part2] step 500/5000 loss=0.009016 coarse=0.009322 fine=0.008084 iter=0.243s rays_per_sec=8,421
[train_nerf_part2] eval step 500/5000 val_psnr=20.361dB best=20.361dB (new best, saved checkpoint) eval=1.91s
[train_nerf_part2] step 750/5000 loss=0.007406 coarse=0.008310 fine=0.006575 iter=0.247s rays_per_sec=8,303
[train_nerf_part2] eval step 750/5000 val_psnr=21.061dB best=21.061dB (new best, saved checkpoint) eval=1.93s
[train_nerf_part2] step 1000/5000 loss=0.007232 coarse=0.007973 fine=0.006434 iter=0.246s rays_per_sec=8,324
[train_nerf_part2] eval step 1000/5000 val_psnr=21.483dB best=21.483dB (new best, saved checkpoint) eval=1.92s
[train_nerf_part2] step 1250/5000 loss=0.006343 coarse=0.007524 fine=0.005590 iter=0.244s rays_per_sec=8,388
[train_nerf_part2] eval step 1250/5000 val_psnr=22.022dB best=22.022dB (new best, saved checkpoint) eval=1.92s
[train_nerf_pa

/tmp/ipykernel_5497/2785096391.py:73: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(d_rgb_uint8, mode="RGB").save(output_path)


Rendering test view 2/60...
Rendering test view 3/60...
Rendering test view 4/60...
Rendering test view 5/60...
Rendering test view 6/60...
Rendering test view 7/60...
Rendering test view 8/60...
Rendering test view 9/60...
Rendering test view 10/60...
Rendering test view 11/60...
Rendering test view 12/60...
Rendering test view 13/60...
Rendering test view 14/60...
Rendering test view 15/60...
Rendering test view 16/60...
Rendering test view 17/60...
Rendering test view 18/60...
Rendering test view 19/60...
Rendering test view 20/60...
Rendering test view 21/60...
Rendering test view 22/60...
Rendering test view 23/60...
Rendering test view 24/60...
Rendering test view 25/60...
Rendering test view 26/60...
Rendering test view 27/60...
Rendering test view 28/60...
Rendering test view 29/60...
Rendering test view 30/60...
Rendering test view 31/60...
Rendering test view 32/60...
Rendering test view 33/60...
Rendering test view 34/60...
Rendering test view 35/60...
Rendering test view 36

{'metrics': {'best_psnr': 23.949074745178223,
  'best_step': 4000,
  'final_loss': 0.005143478512763977,
  'near': 2.015564441680908,
  'far': 6.046693325042725,
  'n_steps': 5000,
  'batch_rays': 2048,
  'n_coarse': 32,
  'n_fine': 32,
  'val_psnr_hist': [18.941973447799683,
   20.360705852508545,
   21.060523986816406,
   21.482524871826172,
   22.021901607513428,
   22.59058713912964,
   23.103837966918945,
   23.248672485351562,
   23.067190647125244,
   23.718221187591553,
   23.748974800109863,
   23.543567657470703,
   23.620059490203857,
   23.731276988983154,
   23.679113388061523,
   23.949074745178223,
   23.69394302368164,
   23.65055561065674,
   23.93770694732666,
   23.891983032226562],
  'eval_steps': [250,
   500,
   750,
   1000,
   1250,
   1500,
   1750,
   2000,
   2250,
   2500,
   2750,
   3000,
   3250,
   3500,
   3750,
   4000,
   4250,
   4500,
   4750,
   5000],
  'total_seconds': 1261.8491126069998,
  'avg_seconds_per_step': 0.25236982252139994,
  'log_ever

In [5]:
import os
import shutil
from google.colab import drive

# Mount Google Drive if it isn't already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

source_dir = 'images'
destination_dir = '/content/drive/MyDrive/CV2/images'

# Copy the directory
if os.path.exists(source_dir):
    # dirs_exist_ok=True allows overwriting/merging into an existing directory
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
    print(f"Successfully copied '{source_dir}' to '{destination_dir}'")
else:
    print(f"Source directory '{source_dir}' does not exist.")

Successfully copied 'images' to '/content/drive/MyDrive/CV2/images'


In [6]:
!pip install viser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.9/252.9 kB 30.2 MB/s eta 0:00:00


In [7]:
# Launch Viser server for camera/ray/sample visualization
import viser
import numpy as np
import torch
import time

# Using the default_cfg defined earlier
cfg = default_cfg

# Load data and setup the rays specifically from the first camera for a clean visualization
images_train, c2ws_train, images_val, c2ws_val, c2ws_test, K = load_data(data_path=cfg["data_path"])
H, W = images_train.shape[1], images_train.shape[2]

# RaysData class internally moves tensors to the given device.
device = "cpu"
images_train_t = torch.tensor(images_train, dtype=torch.float32) if isinstance(images_train, np.ndarray) else images_train
K_t = torch.tensor(K, dtype=torch.float32) if isinstance(K, np.ndarray) else K
c2ws_train_t = torch.tensor(c2ws_train, dtype=torch.float32) if isinstance(c2ws_train, np.ndarray) else c2ws_train

dataset = RaysData(images_train_t, K_t, c2ws_train_t, device=device)

# Get rays of just the first image
rays_o_first_image = dataset.rays_o[:H*W]
rays_d_first_image = dataset.rays_d[:H*W]

# Sample random rays from the first image
num_rays = 100
indices = np.random.randint(low=0, high=H * W, size=num_rays)

rays_o = rays_o_first_image[indices]
rays_d = rays_d_first_image[indices]

# Automatically calculate near and far if overrides are not provided
c2ws_val_t = torch.tensor(c2ws_val, dtype=torch.float32)
near_auto, far_auto = calibrate_near_far_from_cameras(c2ws_val_t)
near = cfg.get("near_override") if cfg.get("near_override") is not None else near_auto
far = cfg.get("far_override") if cfg.get("far_override") is not None else far_auto

# Use the exact argument names from the function
points, t_vals = sample_along_rays(
    ray_origins=rays_o,
    ray_directions=rays_d,
    near=float(near),
    far=float(far),
    n_samples=int(cfg["n_coarse"]),
    perturb=False,
)

# Convert to numpy for viser
rays_o_np = rays_o.cpu().detach().numpy()
rays_d_np = rays_d.cpu().detach().numpy()
points_np = points.cpu().detach().numpy()
images_np = images_train if isinstance(images_train, np.ndarray) else images_train.cpu().detach().numpy()
c2ws_np = c2ws_train if isinstance(c2ws_train, np.ndarray) else c2ws_train.cpu().detach().numpy()
K_np = K if isinstance(K, np.ndarray) else K.cpu().detach().numpy()

# In Colab, share=True is required to tunnel the localhost port to a public URL
server = viser.ViserServer(port=8080, share=True)

fov = float(2 * np.arctan2(H / 2, K_np[0, 0]))
aspect = float(W / H)

# Add all cameras
for i, (image, c2w) in enumerate(zip(images_np, c2ws_np)):
    image_uint8 = (np.clip(image, 0.0, 1.0) * 255.0).astype(np.uint8)
    server.scene.add_camera_frustum(
        f"/cameras/{i}",
        fov=fov,
        aspect=aspect,
        scale=0.15,
        wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
        position=c2w[:3, 3],
        image=image_uint8,
    )

# Add rays
for i, (o, d) in enumerate(zip(rays_o_np, rays_d_np)):
    positions = np.stack((o, o + d * float(far)))
    server.scene.add_spline_catmull_rom(
        f"/rays/{i}",
        positions=positions,
    )

# Add point cloud (samples)
server.scene.add_point_cloud(
    "/samples",
    colors=np.zeros_like(points_np).reshape(-1, 3),
    points=points_np.reshape(-1, 3),
    point_size=0.03,
)

print("Viser server starting. Check the output above for the public share URL.")
print("Stop the cell execution to close the server.")

╭────── viser (listening *:8081) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8081   │
│   Websocket │ ws://localhost:8081     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://5-map.share.viser.studio

Viser server starting. Check the output above for the public share URL.
Stop the cell execution to close the server.
